# 03 — Train a 5-second JSBSim skill Transformer

This notebook consumes the canonical Parquet trajectories and label map produced by
`02_generate_jsbsim_skill_dataset.ipynb`. Complete flights are assigned to an
approximately **0.6:0.2:0.2 train:test:validation split**, stratified by the complete
set of tactical skills demonstrated in each flight.

Only `MODEL_FEATURE_COLUMNS` are model inputs. Commanded skills, native actions, and
other privileged columns are targets or audit metadata only. Transition-spanning
windows are excluded by default; set `BVR_KEEP_MIXED_WINDOWS=1` to label them by their
final sample.

The Transformer is trained on the training split. In accordance with this experiment's
selection protocol, test loss controls checkpointing and early stopping. The validation
split remains untouched until the selected checkpoint receives its final evaluation.
MLflow records configuration, per-epoch metrics, the checkpoint, and the complete
reproducibility bundle.

## 1. Imports, reproducibility, and MLflow configuration

The dataset path exactly matches notebook 02's default output. Environment variables
allow short, reproducible experiments without editing cells. CPU is the safe default;
set `BVR_TRAIN_DEVICE=cuda` only after verifying the local CUDA installation.

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as pads
import pyarrow.parquet as pq
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from torch.utils.data import DataLoader, TensorDataset

from bvr_behavior_prediction.data.observable_columns import MODEL_FEATURE_COLUMNS
from bvr_behavior_prediction.data.privileged_columns import PRIVILEGED_COLUMNS

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
DATASET_DIR = Path(os.getenv(
    "BVR_TRAIN_DATASET",
    REPO_ROOT / "artifacts/datasets/bvr_f16_1v1_jsbsim_skills_v001",
))
OUTPUT_DIR = Path(os.getenv(
    "BVR_CLASSIFIER_OUTPUT", REPO_ROOT / "artifacts/models/jsbsim_skill_transformer_v001"
))
MLFLOW_DB_PATH = (REPO_ROOT / "artifacts/mlflow.db").resolve()
MLFLOW_DB_PATH.parent.mkdir(parents=True, exist_ok=True)
MLFLOW_TRACKING_URI = os.getenv(
    "BVR_MLFLOW_TRACKING_URI", f"sqlite:///{MLFLOW_DB_PATH.as_posix()}"
)
MLFLOW_EXPERIMENT = os.getenv("BVR_MLFLOW_EXPERIMENT", "jsbsim-skill-transformer")
WINDOW_S = 5.0
STRIDE_S = float(os.getenv("BVR_WINDOW_STRIDE_S", "1.0"))
KEEP_MIXED_WINDOWS = os.getenv("BVR_KEEP_MIXED_WINDOWS", "0") == "1"
BATCH_SIZE = int(os.getenv("BVR_BATCH_SIZE", "64"))
MICRO_BATCH_SIZE = int(os.getenv("BVR_MICRO_BATCH_SIZE", str(BATCH_SIZE)))
ACCUMULATION_STEPS = math.ceil(BATCH_SIZE / MICRO_BATCH_SIZE)
GRADIENT_CHECKPOINTING = os.getenv("BVR_GRADIENT_CHECKPOINTING", "1") == "1"
PREPROCESS_CHUNK_WINDOWS = int(os.getenv("BVR_PREPROCESS_CHUNK_WINDOWS", "8192"))
PREPROCESS_WORKERS = int(os.getenv(
    "BVR_PREPROCESS_WORKERS", str(min(12, os.cpu_count() or 1))
))
WINDOW_WRITE_BUFFER_MB = int(os.getenv("BVR_WINDOW_WRITE_BUFFER_MB", "64"))
WINDOW_BUILD_WORKERS = int(os.getenv(
    "BVR_WINDOW_BUILD_WORKERS", str(min(16, os.cpu_count() or 1))
))
ARROW_BATCH_ROWS = int(os.getenv("BVR_ARROW_BATCH_ROWS", "65536"))
METADATA_SCAN_WORKERS = int(os.getenv(
    "BVR_METADATA_SCAN_WORKERS", str(min(12, os.cpu_count() or 1))
))
METADATA_BATCH_ROWS = int(os.getenv("BVR_METADATA_BATCH_ROWS", "4096"))
KEEP_WINDOW_CACHE = os.getenv("BVR_KEEP_WINDOW_CACHE", "1") == "1"
DATALOADER_WORKERS = int(os.getenv("BVR_DATALOADER_WORKERS", "0"))
USE_AMP = os.getenv("BVR_MIXED_PRECISION", "1") == "1"
TORCH_THREADS = int(os.getenv("BVR_TORCH_THREADS", str(min(12, os.cpu_count() or 1))))
REQUESTED_DEVICE = os.getenv("BVR_TRAIN_DEVICE", "cpu").strip().lower()
EPOCHS = int(os.getenv("BVR_TRAIN_EPOCHS", "30"))
PATIENCE = int(os.getenv("BVR_EARLY_STOPPING_PATIENCE", "6"))
LEARNING_RATE = float(os.getenv("BVR_LEARNING_RATE", "0.001"))
D_MODEL = int(os.getenv("BVR_TRANSFORMER_D_MODEL", "128"))
NHEAD = int(os.getenv("BVR_TRANSFORMER_HEADS", "8"))
NUM_LAYERS = int(os.getenv("BVR_TRANSFORMER_LAYERS", "3"))
DROPOUT = float(os.getenv("BVR_TRANSFORMER_DROPOUT", "0.2"))
SEED = int(os.getenv("BVR_TRAIN_SEED", "20260911"))
SPLIT_FRACTIONS = {"train": 0.60, "test": 0.20, "validation": 0.20}

if min(
    BATCH_SIZE, MICRO_BATCH_SIZE, PREPROCESS_CHUNK_WINDOWS, WINDOW_WRITE_BUFFER_MB,
    ARROW_BATCH_ROWS, WINDOW_BUILD_WORKERS,
) < 1 or DATALOADER_WORKERS < 0:
    raise ValueError("Batch size must be positive and data-loader workers cannot be negative")
if BATCH_SIZE % MICRO_BATCH_SIZE or MICRO_BATCH_SIZE > BATCH_SIZE:
    raise ValueError("BVR_MICRO_BATCH_SIZE must divide BVR_BATCH_SIZE and cannot exceed it")
if METADATA_SCAN_WORKERS < 1:
    raise ValueError("BVR_METADATA_SCAN_WORKERS must be at least 1")
if METADATA_BATCH_ROWS < 1:
    raise ValueError("BVR_METADATA_BATCH_ROWS must be at least 1")
if PREPROCESS_WORKERS < 1:
    raise ValueError("BVR_PREPROCESS_WORKERS must be at least 1")
if TORCH_THREADS < 1:
    raise ValueError("BVR_TORCH_THREADS must be at least 1")
if D_MODEL % NHEAD:
    raise ValueError("BVR_TRANSFORMER_D_MODEL must be divisible by BVR_TRANSFORMER_HEADS")
if REQUESTED_DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        f"BVR_TRAIN_DEVICE={REQUESTED_DEVICE!r} requires CUDA, but this PyTorch "
        "installation cannot access it. Use BVR_TRAIN_DEVICE=cpu or install a "
        "CUDA-compatible PyTorch build."
    )

torch.set_num_threads(TORCH_THREADS)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if REQUESTED_DEVICE.startswith("cuda"):
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device(REQUESTED_DEVICE)
AMP_ENABLED = USE_AMP and DEVICE.type == "cuda"
if DEVICE.type == "cuda":
    # TF32 speeds supported matrix multiplications without increasing memory use.
    torch.set_float32_matmul_precision("high")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print({"dataset": str(DATASET_DIR), "device": str(DEVICE), "seed": SEED,
       "mlflow_tracking_uri": mlflow.get_tracking_uri()})

In [ ]:
print(f'METADATA_SCAN_WORKERS  : {METADATA_SCAN_WORKERS}  \n'  +
f'DATALOADER_WORKERS  : {DATALOADER_WORKERS}  \n' +
f'PREPROCESS_WORKERS   : {PREPROCESS_WORKERS}  \n'  +
f'TORCH_THREADS   : {TORCH_THREADS}  \n')

## 2. Load dataset metadata without materialising trajectories

The preferred cold path reads the compact `episodes.parquet` index plus Parquet footer row
counts. For datasets produced by notebook 02, each episode summary already contains its skill
schedule and duration, so decompressing the three metadata columns from every trajectory row is
unnecessary. Only the first two `time_s` values are decoded to establish cadence, trajectory footers are
read concurrently, and the episode index is consumed in bounded record batches. This path is
usually limited by footer reads and keeps peak memory proportional to the required episode
summaries plus one small batch, not the trajectory dataset.

If the compact index is absent or incompatible, the fallback scan is parallelized by Parquet
shard. Each worker projects only the three metadata columns and streams bounded record batches,
so the additional peak memory is roughly one small Arrow batch plus one episode per worker—not a
copy of the dataset. Set `BVR_METADATA_SCAN_WORKERS` to match storage throughput (the default is
at most four), and tune the compact-index batch with `BVR_METADATA_BATCH_ROWS` (default 4096).
Set `BVR_FORCE_METADATA_SCAN=1` to run the validating fallback deliberately.

The resulting episode summaries, sample count, and cadence are saved atomically in a
fingerprinted JSON cache. The fingerprint covers trajectory and episode-index file identity, the
label map, and relevant schema settings, so unchanged future runs skip all Parquet work. The
compact-index path is commonly an order of magnitude faster than a full cold scan, while warm
cache loads are commonly **10–100× or more** faster. The cell reports its actual path and elapsed
time so performance can be checked on the local storage.


In [ ]:
shards = sorted((DATASET_DIR / "trajectories").glob("*.parquet"))
label_map_path = DATASET_DIR / "label_map.json"
if not shards or not label_map_path.exists():
    raise FileNotFoundError(
        f"Expected trajectories/*.parquet and label_map.json under {DATASET_DIR}. "
        "Run notebooks/02_generate_jsbsim_skill_dataset.ipynb first or set BVR_TRAIN_DATASET."
    )

LABELS = json.loads(label_map_path.read_text())["tactical"]
label_to_index = {label: index for index, label in enumerate(LABELS)}
required_columns = ["episode_id", "time_s", "tactical_label", *MODEL_FEATURE_COLUMNS]
trajectory_dataset = pads.dataset(shards, format="parquet")
missing = set(required_columns).difference(trajectory_dataset.schema.names)
assert not missing, f"Missing required columns: {sorted(missing)}"
assert set(MODEL_FEATURE_COLUMNS).isdisjoint(PRIVILEGED_COLUMNS)

METADATA_CACHE_VERSION = 2
METADATA_CACHE_PATH = Path(os.getenv(
    "BVR_METADATA_CACHE", OUTPUT_DIR.parent / f".{OUTPUT_DIR.name}_metadata.json"
))
episodes_path = DATASET_DIR / "episodes.parquet"


def file_identity(path):
    if not path.exists():
        return None
    stat = path.stat()
    return {"name": path.name, "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}


metadata_fingerprint = hashlib.sha256(json.dumps({
    "version": METADATA_CACHE_VERSION,
    "dataset_dir": str(DATASET_DIR.resolve()),
    "shards": [
        {"name": shard.name, "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}
        for shard in shards for stat in [shard.stat()]
    ],
    "episodes": file_identity(episodes_path),
    "label_map_sha256": hashlib.sha256(label_map_path.read_bytes()).hexdigest(),
    "columns": ["episode_id", "time_s", "tactical_label"],
}, sort_keys=True).encode()).hexdigest()


def iter_dataset_episodes(dataset, columns):
    """Yield contiguous episodes while holding at most one scan batch plus one episode."""
    scanner = dataset.scanner(
        columns=columns, batch_size=ARROW_BATCH_ROWS, use_threads=True,
        batch_readahead=4, fragment_readahead=2,
    )
    pending_id, pending = None, {column: [] for column in columns}
    try:
        for batch in scanner.to_batches():
            arrays = {
                column: batch.column(column).to_numpy(zero_copy_only=False)
                for column in columns
            }
            ids = arrays["episode_id"]
            # Append the batch end to the actual ID changes. This avoids the two
            # temporary arrays created by np.r_ while preserving streaming order.
            boundaries = np.flatnonzero(ids[1:] != ids[:-1]) + 1
            left = 0
            for boundary_index in range(len(boundaries) + 1):
                right = boundaries[boundary_index] if boundary_index < len(boundaries) else len(ids)
                episode_id = ids[left]
                if pending_id is not None and episode_id != pending_id:
                    yield pending_id, {
                        column: np.concatenate(parts) if len(parts) > 1 else parts[0]
                        for column, parts in pending.items()
                    }
                    pending = {column: [] for column in columns}
                pending_id = episode_id
                for column in columns:
                    pending[column].append(arrays[column][left:right])
                left = right
        if pending_id is not None:
            yield pending_id, {
                column: np.concatenate(parts) if len(parts) > 1 else parts[0]
                for column, parts in pending.items()
            }
    except pa.ArrowKeyError as error:
        raise RuntimeError(
            "PyArrow's legacy extension registry is incompatible with this pandas session. "
            "Install the project dependencies (which require pyarrow>=14.0.1), restart the "
            "notebook kernel, and run all cells again."
        ) from error


def iter_episodes(columns):
    return iter_dataset_episodes(trajectory_dataset, columns)


def scan_metadata_shard(shard):
    """Summarize one shard; generated episodes never cross shard boundaries."""
    rows, shard_samples, shard_cadence = [], 0, None
    shard_dataset = pads.dataset([shard], format="parquet")
    for episode_id, episode in iter_dataset_episodes(
        shard_dataset, ["episode_id", "time_s", "tactical_label"]
    ):
        times = episode["time_s"]
        deltas = np.diff(times)
        episode_dt = float(np.median(deltas))
        assert episode_dt > 0 and np.allclose(
            deltas, episode_dt, atol=1e-6
        ), "Irregular sample cadence"
        if shard_cadence is not None:
            assert math.isclose(
                episode_dt, shard_cadence, abs_tol=1e-6
            ), "Inconsistent episode cadence"
        shard_cadence = episode_dt
        skills = tuple(sorted(set(episode["tactical_label"])))
        unknown = set(skills).difference(label_to_index)
        assert not unknown, f"Labels absent from label_map.json: {sorted(unknown)}"
        rows.append({"episode_id": episode_id, "skills": skills, "samples": len(times)})
        shard_samples += len(times)
    return rows, shard_samples, shard_cadence


def load_compact_metadata():
    """Use the canonical episode index and Parquet footers instead of a trajectory scan."""
    if os.getenv("BVR_FORCE_METADATA_SCAN", "0") == "1" or not episodes_path.exists():
        return None

    required_episode_columns = {"episode_id", "episode_duration_s", "skill_schedule_json"}
    episode_file = pq.ParquetFile(episodes_path)
    if not required_episode_columns.issubset(episode_file.schema_arrow.names):
        return None

    # A two-row batch recovers cadence without materialising an entire row group. Footer
    # counts provide the exact total and are independent I/O, so read them concurrently.
    first_file = pq.ParquetFile(shards[0])
    first_batch = next(first_file.iter_batches(batch_size=2, columns=["time_s"]), None)
    if first_batch is None or first_batch.num_rows < 2:
        return None
    first_time_values = first_batch.column("time_s").to_numpy(zero_copy_only=False)
    cadence = float(first_time_values[1] - first_time_values[0])
    if cadence <= 0:
        return None

    def parquet_row_count(shard):
        return pq.ParquetFile(shard).metadata.num_rows

    worker_count = min(METADATA_SCAN_WORKERS, len(shards))
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        sample_count = sum(executor.map(parquet_row_count, shards))

    # Keep only the summaries needed downstream. Iterating the compact index avoids the
    # temporary full Arrow table and full Python-record copy created by Table.to_pylist().
    episode_columns = sorted(required_episode_columns)
    rows = []
    for batch in episode_file.iter_batches(
        batch_size=METADATA_BATCH_ROWS, columns=episode_columns, use_threads=True
    ):
        columns = batch.to_pydict()
        records = zip(
            columns["episode_id"],
            columns["episode_duration_s"],
            columns["skill_schedule_json"],
        )
        for episode_id, episode_duration_s, schedule_json in records:
            schedule = json.loads(schedule_json)
            skills = tuple(sorted({schedule["primary"], schedule["secondary"]}))
            unknown = set(skills).difference(label_to_index)
            if unknown:
                raise ValueError(f"Labels absent from label_map.json: {sorted(unknown)}")
            samples = int(round(float(episode_duration_s) / cadence)) + 1
            rows.append({"episode_id": episode_id, "skills": skills, "samples": samples})

    if sum(row["samples"] for row in rows) != sample_count:
        # A non-canonical dataset may have early termination or variable cadence; retain
        # the complete validating scan as a safe compatibility path.
        return None
    if len({row["episode_id"] for row in rows}) != len(rows):
        raise ValueError("Episode IDs must be unique in episodes.parquet")
    return {"episode_rows": rows, "sample_count": sample_count, "cadence": cadence}


def load_metadata_cache():
    if not METADATA_CACHE_PATH.exists():
        return None
    cached = json.loads(METADATA_CACHE_PATH.read_text())
    if cached.get("fingerprint") != metadata_fingerprint:
        return None
    cached["episode_rows"] = [
        {**row, "skills": tuple(row["skills"])} for row in cached["episode_rows"]
    ]
    return cached


scan_started = time.perf_counter()
cached_metadata = load_metadata_cache()
if cached_metadata is None:
    compact_metadata = load_compact_metadata()
else:
    compact_metadata = None
if compact_metadata is not None:
    episode_rows = compact_metadata["episode_rows"]
    sample_count = compact_metadata["sample_count"]
    cadence = compact_metadata["cadence"]
    cached_metadata = {"fingerprint": metadata_fingerprint, **compact_metadata}
    metadata_source = "compact episode index + Parquet footers"
elif cached_metadata is None:
    worker_count = min(METADATA_SCAN_WORKERS, len(shards))
    episode_rows, sample_count, shard_cadences = [], 0, []
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        # Consume results as they arrive in shard order instead of retaining a
        # second, nested copy of every summary until the scan has completed.
        for rows, count, shard_cadence in executor.map(scan_metadata_shard, shards):
            episode_rows.extend(rows)
            sample_count += count
            if shard_cadence is not None:
                shard_cadences.append(shard_cadence)
    if not shard_cadences:
        raise ValueError("No trajectory samples were found")
    cadence = shard_cadences[0]
    assert all(
        math.isclose(value, cadence, abs_tol=1e-6) for value in shard_cadences
    ), "Inconsistent episode cadence"
    if len({row["episode_id"] for row in episode_rows}) != len(episode_rows):
        raise ValueError("Episode IDs must be unique across Parquet shards")
    cached_metadata = {
        "fingerprint": metadata_fingerprint,
        "episode_rows": episode_rows,
        "sample_count": sample_count,
        "cadence": cadence,
    }
    metadata_source = f"parallel scan ({worker_count} workers)"
else:
    episode_rows = cached_metadata["episode_rows"]
    sample_count = int(cached_metadata["sample_count"])
    cadence = float(cached_metadata["cadence"])
    metadata_source = "persistent cache"

if metadata_source != "persistent cache":
    METADATA_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_cache_path = METADATA_CACHE_PATH.with_suffix(METADATA_CACHE_PATH.suffix + ".tmp")
    temporary_cache_path.write_text(json.dumps(cached_metadata, separators=(",", ":")))
    os.replace(temporary_cache_path, METADATA_CACHE_PATH)

SAMPLE_DT_S = cadence
WINDOW_SAMPLES = int(round(WINDOW_S / SAMPLE_DT_S))
STRIDE_SAMPLES = max(1, int(round(STRIDE_S / SAMPLE_DT_S)))
assert WINDOW_SAMPLES >= 2
metadata_elapsed_s = time.perf_counter() - scan_started
print(
    f"Loaded {sample_count:,} samples from {len(episode_rows):,} episodes via "
    f"{metadata_source} in {metadata_elapsed_s:.3f}s"
)


In [ ]:
# Demonstrate that a later notebook run can reload the persisted outputs without rescanning.
reloaded_metadata = load_metadata_cache()
assert reloaded_metadata is not None
assert reloaded_metadata["sample_count"] == sample_count
assert math.isclose(reloaded_metadata["cadence"], SAMPLE_DT_S, abs_tol=1e-12)
assert reloaded_metadata["episode_rows"] == episode_rows
print(
    f"Reloaded cached metadata for {len(reloaded_metadata['episode_rows']):,} episodes from "
    f"{METADATA_CACHE_PATH.resolve()}"
)
del reloaded_metadata


## 3. Stratify complete episodes 60:20:20 by demonstrated skills

A flight can demonstrate more than one skill. Its stratum is therefore the sorted set
of all tactical labels appearing in that episode, rather than only its first or most
frequent label. A 60/40 stratified split is followed by an equal stratified division of
the remainder. This preserves joint skill combinations while keeping every flight—and
all overlapping windows from it—in exactly one split.

Every demonstrated-skill combination needs at least five episodes to be represented in
all three partitions. The production dataset from notebook 02 readily satisfies this;
a deliberately tiny smoke dataset fails with an actionable message rather than silently
falling back to an unstratified split.

In [ ]:
episode_skills = pd.DataFrame.from_records(episode_rows)
episode_skills["stratum"] = episode_skills["skills"].map("|".join)
stratum_counts = episode_skills["stratum"].value_counts()
if len(episode_skills) < 5 or (stratum_counts < 5).any():
    rare = stratum_counts[stratum_counts < 5].to_dict()
    raise ValueError(
        "Stratified 60:20:20 splitting requires at least five flights for every "
        f"demonstrated-skill combination; insufficient strata: {rare}. Generate more flights."
    )

train_episodes, selection_episodes = train_test_split(
    episode_skills,
    test_size=SPLIT_FRACTIONS["test"] + SPLIT_FRACTIONS["validation"],
    random_state=SEED,
    shuffle=True,
    stratify=episode_skills["stratum"],
)
test_episodes, validation_episodes = train_test_split(
    selection_episodes,
    test_size=0.5,
    random_state=SEED,
    shuffle=True,
    stratify=selection_episodes["stratum"],
)
split_tables = {
    "train": train_episodes,
    "test": test_episodes,
    "validation": validation_episodes,
}
split_ids = {name: table["episode_id"].tolist() for name, table in split_tables.items()}
all_split_ids = [set(ids) for ids in split_ids.values()]
assert all(all_split_ids) and not any(
    not all_split_ids[i].isdisjoint(all_split_ids[j])
    for i in range(len(all_split_ids)) for j in range(i + 1, len(all_split_ids))
)
assert set().union(*all_split_ids) == set(episode_skills["episode_id"])

split_audit = pd.concat([
    table.assign(split=name).explode("skills")
    for name, table in split_tables.items()
], ignore_index=True)
split_summary = pd.crosstab(split_audit["skills"], split_audit["split"])
split_summary.loc["TOTAL EPISODES"] = {
    name: len(table) for name, table in split_tables.items()
}
print({name: round(len(ids) / len(episode_skills), 4) for name, ids in split_ids.items()})
display(split_summary)


## 4. Stream five-second windows into disk-backed arrays

After splitting, a label-only streaming pass counts admissible windows. A feature pass
then writes directly into `.npy` memory maps instead of keeping the source trajectories
and all split arrays in RAM. Mixed-window detection is vectorized with a cumulative
change count. Window materialization is also vectorized in bounded chunks, replacing one Python
assignment per window with a small number of bulk memory-map writes. Retained starts normally form
long arithmetic runs; those runs are copied with NumPy basic slices instead of advanced indexing,
removing a full temporary copy of every window batch from the dominant feature-write stage. The
default write batch is at most 64 MiB per writer; tune it with `BVR_WINDOW_WRITE_BUFFER_MB`.
The maps are demand-paged by the OS during training and can therefore be
much larger than host memory without an allocation spike.

Both trajectory passes run independent Parquet shards concurrently. The label-only pass retains
compact per-window selections (32-bit starts, mixed flags, and encoded targets) in memory. The
feature pass therefore does not read the label column, find transitions, filter windows, or encode
targets a second time. This deliberately trades a modest peak-memory increase for less Parquet I/O
and CPU work. The counting pass also precomputes a disjoint output range for every episode,
allowing the feature pass to write directly to the shared memory maps without locks.
`BVR_WINDOW_BUILD_WORKERS` controls feature-scan parallelism; its default of at most eight writers
uses the permitted extra peak memory to overlap more Parquet decoding and bulk memory-map writes.
Feature decoding and materialization are expected to dominate a cold build, so the cell times every
major stage, ranks them by wall time, and explicitly names the measured bottleneck. Completed arrays and
their counts are fingerprinted and persisted atomically; unchanged future runs reopen the `.npy`
files as memory maps and skip both trajectory passes.


In [ ]:
window_cell_started = time.perf_counter()
window_stage_seconds = {}
setup_started = time.perf_counter()

episode_to_split = {
    episode_id: split_name for split_name, ids in split_ids.items() for episode_id in ids
}
CACHE_DIR = Path(os.getenv(
    "BVR_WINDOW_CACHE", OUTPUT_DIR.parent / f".{OUTPUT_DIR.name}_window_cache"
))
WINDOW_CACHE_MANIFEST = CACHE_DIR / "manifest.json"
WINDOW_CACHE_VERSION = 1
window_cache_fingerprint = hashlib.sha256(json.dumps({
    "version": WINDOW_CACHE_VERSION,
    "metadata": metadata_fingerprint,
    "split_ids": split_ids,
    "features": list(MODEL_FEATURE_COLUMNS),
    "labels": LABELS,
    "window_samples": WINDOW_SAMPLES,
    "stride_samples": STRIDE_SAMPLES,
    "keep_mixed": KEEP_MIXED_WINDOWS,
}, sort_keys=True).encode()).hexdigest()
array_specs = {
    "x": (np.float32, (WINDOW_SAMPLES, len(MODEL_FEATURE_COLUMNS))),
    "y": (np.int64, ()),
    "episode_index": (np.int32, ()),
    "start_time_s": (np.float32, ()),
    "end_time_s": (np.float32, ()),
    "mixed_skill": (np.bool_, ()),
}


def load_window_cache():
    """Open a complete, fingerprint-matched cache without loading arrays into RAM."""
    if not WINDOW_CACHE_MANIFEST.exists():
        return None
    try:
        manifest = json.loads(WINDOW_CACHE_MANIFEST.read_text())
        if manifest["fingerprint"] != window_cache_fingerprint:
            return None
        counts = {name: int(value) for name, value in manifest["window_counts"].items()}
        mixed = {name: int(value) for name, value in manifest["mixed_counts"].items()}
        arrays = {
            split_name: {
                key: np.load(CACHE_DIR / f"{split_name}_{suffix}.npy", mmap_mode="r+")
                for key, suffix in {
                    "x": "x", "y": "y", "episode_index": "episode",
                    "start_time_s": "start", "end_time_s": "end", "mixed_skill": "mixed",
                }.items()
            }
            for split_name in split_ids
        }
    except (OSError, ValueError, KeyError, json.JSONDecodeError):
        return None
    if any(len(arrays[name]["y"]) != counts[name] for name in split_ids):
        return None
    return arrays, counts, mixed, manifest


def window_starts(targets):
    starts = np.arange(0, len(targets) - WINDOW_SAMPLES + 1, STRIDE_SAMPLES)
    if not len(starts):
        return starts, np.empty(0, dtype=bool)
    changes = np.empty(len(targets), dtype=np.int32)
    changes[0] = 0
    np.cumsum(targets[1:] != targets[:-1], dtype=np.int32, out=changes[1:])
    mixed = changes[starts + WINDOW_SAMPLES - 1] != changes[starts]
    return starts, mixed


def write_feature_windows(destination, values, starts, output_offset):
    """Bulk-write arithmetic runs using basic slices, without indexing temporaries."""
    if not len(starts):
        return
    windows = np.moveaxis(
        np.lib.stride_tricks.sliding_window_view(values, WINDOW_SAMPLES, axis=0), -1, 1
    )
    bytes_per_window = WINDOW_SAMPLES * values.shape[1] * values.dtype.itemsize
    chunk_windows = max(1, (WINDOW_WRITE_BUFFER_MB * 1024**2) // bytes_per_window)
    # Filtering mixed windows only creates a few gaps around label transitions. Split at
    # those gaps, then use basic strided slices. Advanced indexing (windows[starts]) first
    # copied every selected window into a large temporary before writing the memory map.
    run_edges = np.flatnonzero(np.diff(starts) != STRIDE_SAMPLES) + 1
    run_edges = np.concatenate(([0], run_edges, [len(starts)]))
    for run_index in range(len(run_edges) - 1):
        run_left, run_right = run_edges[run_index:run_index + 2]
        for chunk_left in range(run_left, run_right, chunk_windows):
            chunk_right = min(chunk_left + chunk_windows, run_right)
            first_start = int(starts[chunk_left])
            source_stop = first_start + (chunk_right - chunk_left) * STRIDE_SAMPLES
            output_left = output_offset + chunk_left
            output_right = output_offset + chunk_right
            destination[output_left:output_right] = windows[
                first_start:source_stop:STRIDE_SAMPLES
            ]


window_stage_seconds["setup/fingerprint"] = time.perf_counter() - setup_started
cache_lookup_started = time.perf_counter()
cached_windows = load_window_cache()
window_stage_seconds["cache lookup"] = time.perf_counter() - cache_lookup_started
if cached_windows is not None:
    raw, window_counts, mixed_counts, window_cache_manifest = cached_windows
    print(f"Reused {sum(window_counts.values()):,} cached windows from {CACHE_DIR.resolve()}")
else:
    # Counting shards is independent and I/O-bound. Each worker retains only one bounded
    # Arrow batch and one episode; cap it with the same conservative metadata-worker setting.
    def count_window_shard(shard):
        counts = {name: 0 for name in split_ids}
        mixed = {name: 0 for name in split_ids}
        prepared_episodes = []
        target_dtype = np.min_scalar_type(max(0, len(LABELS) - 1))
        shard_dataset = pads.dataset([shard], format="parquet")
        for episode_id, episode in iter_dataset_episodes(
            shard_dataset, ["episode_id", "tactical_label"]
        ):
            split_name = episode_to_split[episode_id]
            starts, is_mixed = window_starts(episode["tactical_label"])
            mixed[split_name] += int(is_mixed.sum())
            if not KEEP_MIXED_WINDOWS:
                keep = ~is_mixed
                starts, is_mixed = starts[keep], is_mixed[keep]
            # Retain only compact products that the feature pass would otherwise
            # recompute after decoding tactical_label for a second time.
            starts = starts.astype(np.int32, copy=False)
            selected_targets = episode["tactical_label"][
                starts.astype(np.intp, copy=False) + WINDOW_SAMPLES - 1
            ]
            encoded_targets = np.empty(len(selected_targets), dtype=target_dtype)
            matched_targets = np.zeros(len(selected_targets), dtype=bool)
            for label, label_index in label_to_index.items():
                matches = selected_targets == label
                encoded_targets[matches] = label_index
                matched_targets |= matches
            if not matched_targets.all():
                raise ValueError(f"Unknown target labels in episode {episode_id!r}")
            kept_count = len(starts)
            counts[split_name] += kept_count
            prepared_episodes.append(
                (episode_id, split_name, starts, is_mixed, encoded_targets)
            )
        return counts, mixed, prepared_episodes

    worker_count = min(METADATA_SCAN_WORKERS, len(shards))
    count_started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        shard_counts = list(executor.map(count_window_shard, shards))
    window_stage_seconds["label scan/window selection"] = time.perf_counter() - count_started
    print(
        f"[windows] Label scan/window selection: "
        f"{window_stage_seconds['label scan/window selection']:.2f}s "
        f"({worker_count} workers)"
    )

    layout_started = time.perf_counter()
    window_counts = {name: 0 for name in split_ids}
    mixed_counts = {name: 0 for name in split_ids}
    episode_window_layout = {}
    episode_window_selection = {}
    for counts, mixed, prepared_episodes in shard_counts:
        for name in split_ids:
            mixed_counts[name] += mixed[name]
        for episode_id, split_name, starts, is_mixed, encoded_targets in prepared_episodes:
            if episode_id in episode_window_layout:
                raise ValueError(f"Episode {episode_id!r} occurs in more than one Parquet shard")
            count = len(starts)
            episode_window_layout[episode_id] = (split_name, window_counts[split_name], count)
            episode_window_selection[episode_id] = (starts, is_mixed, encoded_targets)
            window_counts[split_name] += count
    del shard_counts
    window_stage_seconds["layout aggregation"] = time.perf_counter() - layout_started
    print(
        f"[windows] Layout aggregation: {window_stage_seconds['layout aggregation']:.2f}s; "
        f"{sum(window_counts.values()):,} retained windows"
    )
    if not all(window_counts.values()):
        raise ValueError("No windows were produced; check episode duration and filtering")

    allocation_started = time.perf_counter()
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    CACHE_DIR.mkdir(parents=True)
    raw = {}
    suffixes = {"x": "x", "y": "y", "episode_index": "episode",
                "start_time_s": "start", "end_time_s": "end", "mixed_skill": "mixed"}
    for split_name, count in window_counts.items():
        raw[split_name] = {
            key: np.lib.format.open_memmap(
                CACHE_DIR / f"{split_name}_{suffixes[key]}.npy", mode="w+", dtype=dtype,
                shape=(count, *tail_shape),
            )
            for key, (dtype, tail_shape) in array_specs.items()
        }

    window_stage_seconds["cache allocation"] = time.perf_counter() - allocation_started
    allocated_gib = sum(
        int(np.prod(array.shape, dtype=np.int64)) * array.dtype.itemsize
        for split_arrays in raw.values() for array in split_arrays.values()
    ) / 1024**3
    print(
        f"[windows] Cache allocation: {window_stage_seconds['cache allocation']:.2f}s "
        f"({allocated_gib:.2f} GiB across memory maps)"
    )

    episode_ids = episode_skills["episode_id"].tolist()
    episode_indices = {episode_id: index for index, episode_id in enumerate(episode_ids)}
    if set(episode_window_layout) != set(episode_ids):
        raise ValueError("Counted Parquet episodes do not match the split episode index")

    feature_columns = ["episode_id", "time_s", *MODEL_FEATURE_COLUMNS]

    def write_window_shard(shard):
        """Decode one shard and write its preassigned, non-overlapping output ranges."""
        written = {name: 0 for name in split_ids}
        shard_dataset = pads.dataset([shard], format="parquet")
        for episode_id, episode in iter_dataset_episodes(shard_dataset, feature_columns):
            split_name, left, expected_count = episode_window_layout[episode_id]
            starts, mixed, encoded_targets = episode_window_selection[episode_id]
            if len(starts) != expected_count:
                raise RuntimeError(f"Window count changed between passes for {episode_id!r}")
            if not len(starts):
                continue
            right = left + len(starts)
            destination = raw[split_name]
            values = np.empty(
                (len(episode["time_s"]), len(MODEL_FEATURE_COLUMNS)), dtype=np.float32
            )
            for feature_index, column in enumerate(MODEL_FEATURE_COLUMNS):
                values[:, feature_index] = episode[column]
            write_feature_windows(destination["x"], values, starts, left)

            destination["y"][left:right] = encoded_targets
            destination["episode_index"][left:right] = episode_indices[episode_id]
            times = episode["time_s"]
            destination["start_time_s"][left:right] = times[starts]
            destination["end_time_s"][left:right] = times[starts + WINDOW_SAMPLES - 1]
            destination["mixed_skill"][left:right] = mixed
            written[split_name] += len(starts)
        return written

    build_workers = min(WINDOW_BUILD_WORKERS, len(shards))
    written_counts = {name: 0 for name in split_ids}
    materialize_started = time.perf_counter()
    with ThreadPoolExecutor(max_workers=build_workers) as executor:
        for shard_written in executor.map(write_window_shard, shards):
            for name in split_ids:
                written_counts[name] += shard_written[name]
    window_stage_seconds["feature scan/window writes"] = time.perf_counter() - materialize_started
    print(
        f"[windows] Feature scan/window writes: "
        f"{window_stage_seconds['feature scan/window writes']:.2f}s "
        f"({build_workers} workers, {sum(written_counts.values()):,} windows)"
    )
    assert written_counts == window_counts
    del episode_window_selection

    flush_started = time.perf_counter()
    for split_name in split_ids:
        for array in raw[split_name].values():
            array.flush()
    window_stage_seconds["memory-map flush"] = time.perf_counter() - flush_started
    print(f"[windows] Memory-map flush: {window_stage_seconds['memory-map flush']:.2f}s")
    window_cache_manifest = {
        "fingerprint": window_cache_fingerprint,
        "window_counts": window_counts,
        "mixed_counts": mixed_counts,
        "normalized": False,
    }
    manifest_started = time.perf_counter()
    temporary_manifest = WINDOW_CACHE_MANIFEST.with_suffix(".tmp")
    temporary_manifest.write_text(json.dumps(window_cache_manifest, indent=2))
    os.replace(temporary_manifest, WINDOW_CACHE_MANIFEST)
    window_stage_seconds["manifest commit"] = time.perf_counter() - manifest_started

# Complete episode assignment already proves that overlapping windows cannot leak.
episode_ids = episode_skills["episode_id"].tolist()
assert len(episode_to_split) == len(set(episode_to_split)) == len(episode_ids)
for split_name in split_ids:
    print(split_name, {"episodes": len(split_ids[split_name]),
                       "windows": window_counts[split_name],
                       "excluded_mixed": mixed_counts[split_name]})
window_total_seconds = time.perf_counter() - window_cell_started
slowest_stage, slowest_seconds = max(window_stage_seconds.items(), key=lambda item: item[1])
print("[windows] Performance summary (wall time):")
for stage_name, elapsed_s in sorted(
    window_stage_seconds.items(), key=lambda item: item[1], reverse=True
):
    print(f"  {stage_name:30s} {elapsed_s:9.2f}s  ({elapsed_s / window_total_seconds:6.1%})")
print(
    f"[windows] Slowest measured stage: {slowest_stage} ({slowest_seconds:.2f}s); "
    f"total cell {window_total_seconds:.2f}s"
)
gc.collect()


In [ ]:
# Demonstrate reloading every product without copying the disk-backed arrays into memory.
reloaded_window_cache = load_window_cache()
assert reloaded_window_cache is not None
reloaded_raw, reloaded_window_counts, reloaded_mixed_counts, reloaded_manifest = reloaded_window_cache
assert reloaded_window_counts == window_counts
assert reloaded_mixed_counts == mixed_counts
assert all(
    reloaded_raw[name][key].shape == raw[name][key].shape
    for name in split_ids for key in array_specs
)
print(
    f"Reloaded {sum(reloaded_window_counts.values()):,} windows across "
    f"{len(reloaded_raw)} splits from {CACHE_DIR.resolve()}"
)
# Continue with the freshly reopened maps, exactly as a future notebook run would.
raw = reloaded_raw
window_cache_manifest = reloaded_manifest
del reloaded_window_cache, reloaded_raw, reloaded_window_counts, reloaded_mixed_counts


## 5. Incremental, in-place preprocessing

`StandardScaler.partial_fit` sees training data in bounded chunks, preventing its
float64 accumulators and temporary reductions from scaling with the full window array.
Normalization also proceeds chunk by chunk and in place. DataLoader workers do not copy
the memory maps, and CUDA pins only individual micro-batches for asynchronous transfer.

Independent split normalization is dispatched concurrently. Each worker receives a fraction of the original chunk size, so the combined active row budget never exceeds `BVR_PREPROCESS_CHUNK_WINDOWS`; parallelism therefore does not raise the configured peak working set. Tune concurrency with `BVR_PREPROCESS_WORKERS`.


In [ ]:
scaler = StandardScaler(copy=False)
if window_cache_manifest.get("normalized", False):
    normalization_mean = np.asarray(window_cache_manifest["normalization_mean"], dtype=np.float32)
    normalization_scale = np.asarray(window_cache_manifest["normalization_scale"], dtype=np.float32)
    scaler.mean_ = normalization_mean.astype(np.float64)
    scaler.scale_ = normalization_scale.astype(np.float64)
    scaler.var_ = scaler.scale_ ** 2
    scaler.n_features_in_ = len(normalization_mean)
    print("Reused normalized window maps and cached scaler parameters")
else:
    train_x = raw["train"]["x"]
    for left in range(0, len(train_x), PREPROCESS_CHUNK_WINDOWS):
        chunk = train_x[left:left + PREPROCESS_CHUNK_WINDOWS]
        scaler.partial_fit(chunk.reshape(-1, chunk.shape[-1]))
    normalization_mean = scaler.mean_.astype(np.float32)
    normalization_scale = scaler.scale_.astype(np.float32)
    del train_x, chunk

    # Normalize independent maps concurrently while dividing the existing row budget
    # among workers. The total active rows never exceeds PREPROCESS_CHUNK_WINDOWS.
    normalization_workers = min(PREPROCESS_WORKERS, len(raw), PREPROCESS_CHUNK_WINDOWS)
    normalization_chunk_windows = max(1, PREPROCESS_CHUNK_WINDOWS // normalization_workers)

    def normalize_split(values):
        x = values["x"]
        for left in range(0, len(x), normalization_chunk_windows):
            chunk = x[left:left + normalization_chunk_windows]
            chunk -= normalization_mean
            chunk /= normalization_scale
        x.flush()

    with ThreadPoolExecutor(max_workers=normalization_workers) as executor:
        tuple(executor.map(normalize_split, raw.values()))
    window_cache_manifest.update({
        "normalized": True,
        "normalization_mean": normalization_mean.tolist(),
        "normalization_scale": normalization_scale.tolist(),
    })
    temporary_manifest = WINDOW_CACHE_MANIFEST.with_suffix(".tmp")
    temporary_manifest.write_text(json.dumps(window_cache_manifest, indent=2))
    os.replace(temporary_manifest, WINDOW_CACHE_MANIFEST)

loaders = {}
for split_name, values in raw.items():
    x = values["x"]
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(values["y"]))
    generator = torch.Generator().manual_seed(SEED) if split_name == "train" else None
    loaders[split_name] = DataLoader(
        dataset, batch_size=MICRO_BATCH_SIZE, shuffle=split_name == "train",
        generator=generator,
        num_workers=DATALOADER_WORKERS, pin_memory=DEVICE.type == "cuda",
        persistent_workers=DATALOADER_WORKERS > 0,
    )
print("Train tensor:", tuple(loaders["train"].dataset.tensors[0].shape),
      "effective batch:", MICRO_BATCH_SIZE * ACCUMULATION_STEPS)


## 6. Memory-efficient Transformer training tracked by MLflow

CUDA uses automatic mixed precision, fused AdamW, TF32, pinned asynchronous transfers,
and PyTorch's optimized attention kernels. Training uses a configurable micro-batch and
gradient accumulation to preserve the requested effective batch size. Activation
checkpointing (on by default) recomputes encoder layers during backward, trading modest
compute for a large reduction in peak activation memory. Disable it with
`BVR_GRADIENT_CHECKPOINTING=0` when throughput matters more than memory, or tune
`BVR_MICRO_BATCH_SIZE` independently of `BVR_BATCH_SIZE`.

After every epoch, **test loss** controls checkpointing and early stopping. MLflow
receives parameters and train/test metrics for every epoch.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length, dropout):
        super().__init__()
        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)
        frequencies = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10_000.0) / d_model)
        )
        encoding = torch.zeros(max_length, d_model)
        encoding[:, 0::2] = torch.sin(positions * frequencies)
        encoding[:, 1::2] = torch.cos(positions * frequencies)
        self.register_buffer("encoding", encoding.unsqueeze(0), persistent=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, sequence):
        return self.dropout(sequence + self.encoding[:, :sequence.size(1)])


class SkillTransformer(nn.Module):
    def __init__(self, input_dim, class_count, window_samples, d_model=128,
                 nhead=8, layers=3, dropout=0.2):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.positions = SinusoidalPositionalEncoding(d_model, window_samples, dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4 * d_model,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=layers, norm=nn.LayerNorm(d_model)
        )
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, class_count))

    def forward(self, sequence):
        encoded = self.positions(self.input_projection(sequence))
        if self.training and GRADIENT_CHECKPOINTING:
            for layer in self.encoder.layers:
                encoded = torch_checkpoint(layer, encoded, use_reentrant=False)
            if self.encoder.norm is not None:
                encoded = self.encoder.norm(encoded)
        else:
            encoded = self.encoder(encoded)
        return self.classifier(encoded.mean(dim=1))


model_config = {
    "input_dim": len(MODEL_FEATURE_COLUMNS), "class_count": len(LABELS),
    "window_samples": WINDOW_SAMPLES, "d_model": D_MODEL, "nhead": NHEAD,
    "layers": NUM_LAYERS, "dropout": DROPOUT,
}
model = SkillTransformer(**model_config).to(DEVICE)
counts = np.bincount(raw["train"]["y"], minlength=len(LABELS))
weights = counts.sum() / (len(LABELS) * np.maximum(counts, 1))
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4, fused=DEVICE.type == "cuda"
)
grad_scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


def run_epoch(loader, training=False):
    model.train(training)
    total = 0
    if DEVICE.type == "cuda":
        # Keep reductions device-side and synchronize only once per epoch rather
        # than calling .item() twice for every accelerator batch.
        totals = torch.zeros(2, dtype=torch.float64, device=DEVICE)
    else:
        total_loss = total_correct = 0
    if training:
        optimizer.zero_grad(set_to_none=True)
    for batch_index, (features, target) in enumerate(loader):
        features = features.to(DEVICE, non_blocking=True)
        target = target.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training), torch.autocast(
            device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED
        ):
            logits = model(features)
            loss = criterion(logits, target)
        if training:
            grad_scaler.scale(loss / ACCUMULATION_STEPS).backward()
            update = (batch_index + 1) % ACCUMULATION_STEPS == 0 or batch_index + 1 == len(loader)
            if update:
                grad_scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                grad_scaler.step(optimizer)
                grad_scaler.update()
                optimizer.zero_grad(set_to_none=True)
        correct = (logits.detach().argmax(1) == target).sum()
        if DEVICE.type == "cuda":
            totals[0] += loss.detach() * len(target)
            totals[1] += correct
        else:
            total_loss += loss.detach().item() * len(target)
            total_correct += correct.item()
        total += len(target)
    if DEVICE.type == "cuda":
        total_loss, total_correct = totals.cpu().tolist()
    return {"loss": total_loss / total, "accuracy": total_correct / total}


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / "checkpoint.pt"
history, best_test_loss, stale_epochs = [], float("inf"), 0
mlflow_params = {
    **model_config, "architecture": "transformer_encoder", "batch_size": BATCH_SIZE,
    "micro_batch_size": MICRO_BATCH_SIZE, "accumulation_steps": ACCUMULATION_STEPS,
    "gradient_checkpointing": GRADIENT_CHECKPOINTING,
    "epochs": EPOCHS, "patience": PATIENCE, "learning_rate": LEARNING_RATE,
    "seed": SEED, "device": str(DEVICE), "mixed_precision": AMP_ENABLED,
    "dataloader_workers": DATALOADER_WORKERS, "window_s": WINDOW_S, "stride_s": STRIDE_S,
    **{f"split_{name}": fraction for name, fraction in SPLIT_FRACTIONS.items()},
}

with mlflow.start_run(run_name=f"transformer-seed-{SEED}") as active_run:
    run_id = active_run.info.run_id
    mlflow.log_params(mlflow_params)
    mlflow.set_tags({"dataset": DATASET_DIR.name, "checkpoint_selection_split": "test"})
    for epoch in range(1, EPOCHS + 1):
        train_metrics = run_epoch(loaders["train"], training=True)
        test_metrics = run_epoch(loaders["test"])
        row = {"epoch": epoch,
               **{f"train_{key}": value for key, value in train_metrics.items()},
               **{f"test_{key}": value for key, value in test_metrics.items()}}
        history.append(row)
        mlflow.log_metrics({key: value for key, value in row.items() if key != "epoch"}, step=epoch)
        print(f"{epoch:02d} train loss={train_metrics['loss']:.4f} "
              f"acc={train_metrics['accuracy']:.3f} test loss={test_metrics['loss']:.4f} "
              f"acc={test_metrics['accuracy']:.3f}")
        if test_metrics["loss"] < best_test_loss - 1e-4:
            best_test_loss = test_metrics["loss"]
            stale_epochs = 0
            torch.save({
                "epoch": epoch, "test_loss": best_test_loss,
                "model_state_dict": model.state_dict(), "model_config": model_config,
            }, CHECKPOINT_PATH)
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print("Early stopping on test loss")
                break

    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    mlflow.log_metric("best_test_loss", checkpoint["test_loss"])
    mlflow.log_metric("best_epoch", checkpoint["epoch"])
    mlflow.log_artifact(str(CHECKPOINT_PATH), artifact_path="checkpoints")

history = pd.DataFrame(history)
history.plot(x="epoch", y=["train_loss", "test_loss"], grid=True,
             title="MLflow-tracked learning curves")
plt.show()
print({"mlflow_run_id": run_id, "selected_epoch": checkpoint["epoch"]})


## 7. Final evaluation on the untouched validation set

Validation is not used for scaling, optimization, checkpoint selection, or early
stopping. It provides the final class-wise report for the test-selected checkpoint.

In [ ]:
def predict(loader):
    model.eval()
    # Exact-size arrays avoid Python-object lists and their final full copies.
    actual = np.empty(len(loader.dataset), dtype=np.int64)
    predicted = np.empty_like(actual)
    offset = 0
    with torch.inference_mode():
        for features, target in loader:
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                logits = model(features.to(DEVICE, non_blocking=True))
            batch_size = len(target)
            actual[offset:offset + batch_size] = target.numpy()
            predicted[offset:offset + batch_size] = logits.argmax(1).cpu().numpy()
            offset += batch_size
    assert offset == len(actual)
    return actual, predicted


y_validation, y_pred = predict(loaders["validation"])
validation_accuracy = float((y_validation == y_pred).mean())
report = classification_report(
    y_validation, y_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    zero_division=0, digits=3, output_dict=True,
)
print(classification_report(
    y_validation, y_pred, labels=np.arange(len(LABELS)), target_names=LABELS,
    zero_division=0, digits=3,
))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_validation, y_pred, labels=np.arange(len(LABELS)), display_labels=LABELS,
    normalize="true", xticks_rotation=45, cmap="Blues", ax=ax,
)
ax.set_title("Untouched validation confusion matrix (row normalized)")
plt.tight_layout()
plt.show()


## 8. Save and log the reproducible bundle

The bundle includes the test-selected Transformer, training-only normalization,
stratified episode assignments, class and feature order, window configuration, history,
and validation report. The same files are attached to the active MLflow run.

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(), "model_config": model_config,
    "feature_columns": list(MODEL_FEATURE_COLUMNS), "labels": LABELS,
    "window_samples": WINDOW_SAMPLES, "sample_dt_s": SAMPLE_DT_S,
    "selected_epoch": checkpoint["epoch"], "selection_test_loss": checkpoint["test_loss"],
}, OUTPUT_DIR / "model.pt")
(OUTPUT_DIR / "preprocessing.json").write_text(json.dumps({
    "feature_columns": list(MODEL_FEATURE_COLUMNS),
    "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
}, indent=2))
(OUTPUT_DIR / "split.json").write_text(json.dumps({
    "seed": SEED, "fractions": SPLIT_FRACTIONS, "stratification": "demonstrated_skill_set",
    "episode_ids": split_ids, "window_s": WINDOW_S, "stride_s": STRIDE_S,
    "keep_mixed_windows": KEEP_MIXED_WINDOWS,
}, indent=2))
history.to_csv(OUTPUT_DIR / "history.csv", index=False)
(OUTPUT_DIR / "validation_report.json").write_text(json.dumps(report, indent=2))
def write_window_metadata(split_name, values):
    """Write metadata incrementally without constructing a multi-million-row DataFrame."""
    import pyarrow.parquet as pq

    destination = OUTPUT_DIR / f"{split_name}_windows.parquet"
    writer = None
    episode_dictionary = pa.array(episode_ids)
    label_dictionary = pa.array(LABELS)
    try:
        for left in range(0, len(values["y"]), PREPROCESS_CHUNK_WINDOWS):
            right = min(left + PREPROCESS_CHUNK_WINDOWS, len(values["y"]))
            indices = values["episode_index"][left:right]
            table = pa.table({
                "episode_id": episode_dictionary.take(pa.array(indices, type=pa.int32())),
                "start_time_s": values["start_time_s"][left:right],
                "end_time_s": values["end_time_s"][left:right],
                "mixed_skill": values["mixed_skill"][left:right],
                "label": label_dictionary.take(pa.array(values["y"][left:right], type=pa.int64())),
            })
            writer = writer or pq.ParquetWriter(destination, table.schema, compression="zstd")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()


for split_name, values in raw.items():
    write_window_metadata(split_name, values)

# Do not upload the potentially enormous temporary memory maps as MLflow artifacts.
bundle_files = ["model.pt", "preprocessing.json", "split.json", "history.csv",
                "validation_report.json", *[f"{name}_windows.parquet" for name in raw]]
with mlflow.start_run(run_id=run_id):
    mlflow.log_metric("validation_accuracy", validation_accuracy)
    for filename in bundle_files:
        mlflow.log_artifact(str(OUTPUT_DIR / filename), artifact_path="training_bundle")
print("Saved training bundle to", OUTPUT_DIR.resolve())

if not KEEP_WINDOW_CACHE:
    # Drop every tensor/memmap view before removing backing files (also works on Windows).
    loaders.clear()
    del loaders, dataset, values, raw
    gc.collect()
    shutil.rmtree(CACHE_DIR)
    print("Removed temporary window cache", CACHE_DIR.resolve())


## Interpretation notes

* Results are episode-held-out, not independent random-window performance.
* The test partition is intentionally a **development selection set** in this protocol;
  report the untouched validation metrics as the final generalization estimate.
* Per-class metrics matter because aggregate accuracy can hide weak rare-skill behavior.
* Mixed windows are better suited to a future transition or multi-label model.
* Synthetic performance does not establish real-world tactical generalization.
## Cell-by-cell acceleration review

Every executable cell was reviewed under the constraint that an optimization must not increase
peak memory. Cells 2 and 5 are already setup/cache verification paths: imports are performed once,
Torch thread use is bounded, and the cache check reads only its compact JSON artifact. Cell 4 now
uses fewer grouping temporaries and consumes parallel scan results incrementally. Cell 7 avoids
materialized set intersections. Cell 9 uses an in-place, 32-bit transition prefix, direct float32
feature assembly, and compact label conversion. Cell 11 creates only the RNG needed by the shuffled
loader and normalizes independent split maps concurrently within the original total chunk budget. Cell 13 removes per-batch CUDA metric synchronizations
while preserving activation checkpointing and AMP. Cell 15 writes predictions directly into
exact-size arrays. Cell 17 uses Arrow `take` rather than Python string-object lists when emitting
metadata.

Markdown cells perform no computation; each was checked for guidance that could accidentally select
a slower or higher-memory path. The documented knobs retain bounded buffers, disk-backed arrays,
CPU-safe defaults, and explicit opt-ins for memory-saving CUDA acceleration.

